In [ ]:
## define water class
import math
a = 0.14772234
OH_length = 0.08724331
H_angle = 103.6 * math.pi/180
H_angle_half = 51.8 * math.pi/180

class OPC:
    def __init__(self, row_list):
        self.coord = {}
        self.row = {}
        self.add_atom(row_list)
        
    def add_atom(self, row_list):
        k = row_list[2]
        # print(k)
        self.coord[k] = np.array([float(coord) for coord in row_list[4:7]])
        self.row[k] = row_list

    def check_coord(self):
        # Is OH bond length correct?
        length1 = np.linalg.norm(self.coord["OW"] - self.coord["HW1"])
        length2 = np.linalg.norm(self.coord["OW"] - self.coord["HW2"])
        HHlength = np.linalg.norm(self.coord["HW1"] - self.coord["HW2"])
        assert length1 > 0.07
        assert length2 > 0.07
        assert length1 < 0.11
        assert length2 < 0.11
        assert HHlength > 0.12 and HHlength < 0.16

    def adjust_coord(self):
        """ MW is the virtual site located between O and the two Hs """
        # Step 1: get the avg coord of the two H
        avgH = self.coord["HW1"] + self.coord["HW2"] / 2
        O2H_dir = avgH - self.coord["OW"]
        O2H_dir_final = OH_length * math.cos(H_angle_half) * O2H_dir/ np.linalg.norm(O2H_dir)
        
        # Step 2: get the avg H to the right bond length
        H2H_dir = self.coord["HW2"] - self.coord["HW1"]
        H2H_dir_final = 2 * OH_length * math.sin(H_angle_half) * H2H_dir/ np.linalg.norm(H2H_dir)

        # Step 3: get the x1 and x2
        newH_avg = self.coord["OW"] + O2H_dir_final
        self.coord["HW1"] = newH_avg + H2H_dir_final / 2
        self.coord["HW2"] = newH_avg - H2H_dir_final / 2
        
        # Step 4: x1 + 2*a*(x3 + x2- 2 * x1)
        self.coord["MW"] = self.coord["OW"] + 2 * a * (self.coord["HW1"] + self.coord["HW2"] - 2 * self.coord["OW"])

        # write the self.row
        for k in ["OW", "HW1", "HW2", "MW"]:
            self.coord[k] = [f"{s:.3f}" for s in list(self.coord[k])]
            self.row[k][4:7] = self.coord[k]

In [ ]:
# step 1: replace multi-space with single-space

new_allrows = []
water = None
with open(f"tip4p.gro", 'r') as tip4p:
    allrows = tip4p.readlines()
    for i, row in enumerate(allrows):
        # while allrows[i].count("  ") > 0:
        #     allrows[i] = allrows[i].replace("  ", " ")
        
        # Header or bottom -> 
        if not allrows[i][5:10].count("SOL") > 0:
            new_allrows.append(row)
            # print(allrows[i])
        else:
            row_list = []
            row_list.append(allrows[i][:5].replace(' ', ''))
            row_list.append(allrows[i][5:10].replace(' ', ''))
            row_list.append(allrows[i][10:15].replace(' ', ''))
            row_list.append(allrows[i][15:20].replace(' ', ''))
            row_list.append(allrows[i][20:28].replace(' ', ''))
            row_list.append(allrows[i][28:36].replace(' ', ''))
            row_list.append(allrows[i][36:44].replace(' ', ''))
            row_list.append(allrows[i][44:52].replace(' ', ''))
            row_list.append(allrows[i][52:60].replace(' ', ''))
            row_list.append(allrows[i][60:68].replace(' ', ''))
            print(row_list)
            # conversion here
            if row_list[2] == "OW":
                water = OPC(row_list)
                # print("OW")
            elif row_list[2] in ["HW1", "HW2", "MW"]:
                water.add_atom(row_list)
                # print(row_list[1])

            # If all coord has been collected?
            if water != None and len(water.coord) == 4:
                water.check_coord()
                water.adjust_coord()

                for k in ["OW", "HW1", "HW2", "MW"]:
                    new_allrows.append(water.row[k])
                    

In [ ]:
## Ensure the right space
# delineation_space = [4, 4, 3, 2, 2, 2, 1, 1, 1]
content_space = [5, 5, 5, 5, 8, 8, 8, 8, 8, 8]
for i, row in enumerate(new_allrows):
    # row = row.split(' ')
    if isinstance(row, list):
        if row[1].count("SOL") > 0:
            new_row = ""
            new_row += " "*(5 - len(row[0])) + row[0]
            new_row += " "*(5 - len(row[1])) + row[1]
            new_row += " "*(5 - len(row[2])) + row[2]
            new_row += " "*(5 - len(row[3])) + row[3]
            new_row += " "*(8 - len(row[4])) + row[4]
            new_row += " "*(8 - len(row[5])) + row[5]
            new_row += " "*(8 - len(row[6])) + row[6]
            new_row += " "*(8 - len(row[7])) + row[7]
            new_row += " "*(8 - len(row[8])) + row[8]
            new_row += " "*(8 - len(row[9])) + row[9]
            new_row += '\n'
            new_allrows[i] = new_row
new_allrows

In [ ]:
with open(f"opc.gro", "a+") as file:
    file.write("".join(new_allrows))

In [3]:
water

NameError: name 'water' is not defined

In [ ]:
new_allrows

In [1]:
from gromacs_script_writer import GromacsScriptWriter
writer = GromacsScriptWriter(
            destination_dir="./",
            dt=0.002,
            dt_em=0.01,
            em_max_steps=50000,
            snapshot_interval=5000
        )

In [2]:
writer.write_all_scripts(temp=300, tot_time=0.02, task="npt")


Writing MDP files to: .
Temperature: 300 K, Production time: 0.02 ns

✓ Written: ions.mdp
✓ Written: minim.mdp
✓ Written: nvt_300K.mdp
✓ Written: npt_300K.mdp

All MDP files written successfully!

